In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# LLM Quantization Tradeoff Analyzer
**Measures:** TTFT · TPS · VRAM · Semantic Quality across FP16 / INT8 / INT4

### Setup
- Enable GPU: Runtime → Change runtime type → GPU (T4 preferred)
- Run all cells in order
- Gradio UI launches in the last cell

In [ ]:
!pip install -q transformers bitsandbytes accelerate gradio sentence-transformers pandas matplotlib seaborn

In [ ]:
pip install bitsandbytes>=0.46.1

In [1]:
# Cell 2: Check GPU
import torch
assert torch.cuda.is_available()
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU  : Tesla T4
VRAM : 14.6 GB


In [2]:
import os, gc, json, time, warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd

PROMPT_SUITE = {
    "reasoning": [
        "If all Bloops are Razzies and all Razzies are Lazzies, are all Bloops definitely Lazzies? Explain step by step.",
        "A bat and ball cost $1.10. The bat costs $1 more than the ball. How much does the ball cost? Show your reasoning.",
        "Three friends split a restaurant bill. Alice pays twice what Bob pays. Bob pays $5 more than Carol. Total is $85. How much does each pay?",
        "If it takes 5 machines 5 minutes to make 5 widgets, how long does it take 100 machines to make 100 widgets?",
        "There are 23 people in a room. What is the probability two share a birthday? Is it above or below 50%? Explain intuitively.",
    ],
    "summarization": [
        "Summarize the key differences between TCP and UDP in under 100 words using a markdown table.",
        "Explain transformer attention mechanism to a software engineer in 3 bullet points.",
        "Summarize what quantization is and why it matters for LLM deployment in 4 sentences.",
        "Write a concise technical summary of gradient descent with momentum vs Adam optimizer.",
        "Summarize the CAP theorem for a distributed systems beginner in plain English, under 80 words.",
    ],
    "structured_output": [
        'Return a JSON object with keys: name, category, risk_level for "deploying INT4 quantized models in production".',
        "Generate a YAML config for a FastAPI service with fields: host, port, workers, log_level, model_path.",
        'Return a JSON array of 3 ML hyperparameters with fields: name, type, typical_range, effect_on_training.',
        'Output a markdown table comparing GPTQ, AWQ, and GGUF quantization formats across: speed, quality, tooling.',
        'Return JSON: {"steps": [...]} for fine-tuning a small LLM on domain data. Each step: name, description.',
    ],
    "sql": [
        "Write a SQL query to find the top 5 customers by total order value from tables: orders(id, customer_id, amount) and customers(id, name).",
        "Write a SQL window function to compute a 7-day rolling average of daily_sales from a sales table.",
        "Write a SQL query to find duplicate emails in a users table, showing email and count.",
        "Given tables: products(id, name, price) and inventory(product_id, quantity), write SQL to find products with quantity below 10.",
        "Write a SQL CTE that finds the second highest salary per department from an employees table.",
    ],
    "long_context": [
        "Given this context: 'The Attention mechanism was introduced in 2014 by Bahdanau et al., later refined in the 2017 Transformer paper by Vaswani et al.' — Who introduced attention and when?",
        "Context: 'RLHF stands for Reinforcement Learning from Human Feedback. It was popularized by InstructGPT (2022) and is used to align LLMs with human preferences.' — What year was RLHF popularized for LLMs?",
        "Context: 'FP16 uses 2 bytes per weight, INT8 uses 1 byte, INT4 uses 0.5 bytes.' — How many GB of VRAM does a 7B parameter model need in each precision?",
        "Context: 'LoRA adds trainable rank decomposition matrices to frozen weights. It was introduced by Hu et al. in 2021.' — What is the core idea of LoRA and who proposed it?",
        "Context: 'Speculative decoding uses a small draft model to propose tokens verified by the larger model.' — What is the role of the draft model in speculative decoding?",
    ],
    "hallucination_sensitive": [
        "Who is the CEO of Anthropic? Answer only if you are certain.",
        "What is the exact parameter count of Llama 3.1 8B? Be precise.",
        "When exactly was GPT-4 released? Provide only the date you are confident about.",
        "What is the context window of Mistral 7B v0.1? If unsure, say so.",
        "Name the paper that introduced the Mixture of Experts architecture for LLMs. Be specific.",
    ],
}

In [3]:

# ── Model Loader ───────────────────────────────────────────────────────────────

class ModelLoader:
    SUPPORTED_MODELS = {
        "TinyLlama-1.1B": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "Phi-3-mini":     "microsoft/Phi-3-mini-4k-instruct",
        "Qwen2.5-3B":     "Qwen/Qwen2.5-3B-Instruct",
    }

    def __init__(self, model_key: str = "TinyLlama-1.1B"):
        self.model_key    = model_key
        self.model_id     = self.SUPPORTED_MODELS[model_key]
        self.model        = None
        self.tokenizer    = None
        self._model_vram_mb = 0.0
        self._baseline_vram_bytes = 0

    def load(self, precision: str):
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

        print(f"\n{'='*60}\nLoading {self.model_key} @ {precision}\n{'='*60}")

        # unload handles: del model/tokenizer, gc.collect,
        # empty_cache, reset_peak_memory_stats — single place for all cleanup
        self.unload()

        # Record the clean baseline AFTER unload — this is the process overhead
        # (CUDA context, PyTorch allocator reserved blocks, etc.) that should
        # NOT be charged to the model. Subtract this from post-load allocation.
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            self._baseline_vram_bytes = torch.cuda.memory_allocated()

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id, trust_remote_code=True)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        common_kwargs = dict(device_map="auto", trust_remote_code=True)

        if precision == "FP16":
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_id, dtype=torch.float16, **common_kwargs)
        elif precision == "INT8":
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_id,
                quantization_config=BitsAndBytesConfig(load_in_8bit=True),
                **common_kwargs)
        elif precision == "INT4":
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_id,
                quantization_config=BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type="nf4",
                ),
                **common_kwargs)
        else:
            raise ValueError(f"Unknown precision '{precision}'. Choose from: FP16, INT8, INT4")

        self.model.eval()

        # Snapshot VRAM *delta*: measure before-load baseline then subtract.
        # memory_allocated() returns all GPU memory in the process — without
        # the delta approach, residual bitsandbytes calibration buffers and
        # INT→FP16 dequant scratch tensors from the loading phase inflate the
        # INT8/INT4 readings, making quantized models appear *larger* than FP16.
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            self._model_vram_mb = (
                torch.cuda.memory_allocated() - self._baseline_vram_bytes
            ) / 1024**2
        else:
            self._model_vram_mb = 0.0

        print(f"Model VRAM: {self._model_vram_mb:.1f} MB")

    def unload(self):
        if self.model is not None:
            del self.model
            self.model = None
        if self.tokenizer is not None:
            del self.tokenizer
            self.tokenizer = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()          # finish any pending GPU ops
            torch.cuda.empty_cache()          # release unused cached memory back to OS
            torch.cuda.reset_peak_memory_stats()  # reset watermark for next precision
        self._baseline_vram_bytes = 0



In [4]:
# ── Inference Runner ───────────────────────────────────────────────────────────

def _apply_chat_template(tokenizer, prompt: str) -> str:
    if not hasattr(tokenizer, "apply_chat_template") or tokenizer.chat_template is None:
        return prompt
    messages = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


class InferenceRunner:
    def __init__(self, model, tokenizer, max_new_tokens: int = 200):
        self.model          = model
        self.tokenizer      = tokenizer
        self.max_new_tokens = max_new_tokens
        self.device         = next(model.parameters()).device

    def run(self, prompt: str) -> dict:
        from transformers import LogitsProcessor, LogitsProcessorList

        formatted = _apply_chat_template(self.tokenizer, prompt)
        inputs    = self.tokenizer(
            formatted, return_tensors="pt", return_attention_mask=True
        ).to(self.device)
        input_len         = inputs["input_ids"].shape[1]
        first_token_time  = [None]

        class TTFTProbe(LogitsProcessor):
            def __init__(self_inner):
                self_inner.step = 0
            def __call__(self_inner, input_ids, scores):
                if self_inner.step == 0:
                    if torch.cuda.is_available():
                        torch.cuda.synchronize()
                    first_token_time[0] = time.perf_counter()
                self_inner.step += 1
                return scores

        with torch.no_grad():
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t_start = time.perf_counter()

            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                eos_token_id=self.tokenizer.eos_token_id,
                pad_token_id=self.tokenizer.pad_token_id,
                logits_processor=LogitsProcessorList([TTFTProbe()]),
            )

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t_end = time.perf_counter()

        generated_ids    = outputs[0][input_len:]
        response         = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        generated_tokens = len(generated_ids)
        total_latency_ms = (t_end - t_start) * 1000
        ttft_ms          = (first_token_time[0] - t_start) * 1000 if first_token_time[0] else total_latency_ms
        tps              = generated_tokens / (t_end - t_start) if (t_end - t_start) > 0 else 0.0

        return {
            "ttft_ms":          round(ttft_ms, 2),
            "tps":              round(tps, 2),
            "total_latency_ms": round(total_latency_ms, 2),
            "generated_tokens": generated_tokens,
            "response":         response,
        }


In [5]:

# ── Metrics Collector ──────────────────────────────────────────────────────────

class MetricsCollector:

    @staticmethod
    def benchmark_performance(runner: InferenceRunner, n_runs: int = 5) -> dict:
        """
        Run a fixed prompt N times and return averaged inference stats.
        Pass 1 is always discarded (JIT + cold cache warmup cost).
        If TTFT is climbing across passes (thermal throttle signal),
        uses the stable middle window instead of all warm passes.

        Note: uses a single fixed-length prompt so TTFT reflects one
        specific prefill length (~20 tokens). Not representative of
        variable-length production traffic — state this as a known limitation.
        """
        PROMPT = (
            "Explain the difference between model quantization and model pruning "
            "for LLM inference optimization. Be concise."
        )
        results = []

        print(f"  Running {n_runs} inference passes...")
        for i in range(n_runs):
            r = runner.run(PROMPT)
            results.append(r)
            print(f"    Pass {i+1}: TTFT={r['ttft_ms']:.1f}ms  TPS={r['tps']:.1f}  Tokens={r['generated_tokens']}")

        # detect thermal throttle: if second half of passes is >10% slower
        # than first half, use only the stable middle window for averaging
        ttfts = [r["ttft_ms"] for r in results]
        if len(ttfts) >= 3:
            mid             = len(ttfts) // 2
            first_half_mean = sum(ttfts[:mid]) / mid
            last_half_mean  = sum(ttfts[mid:]) / (len(ttfts) - mid)
            if last_half_mean > first_half_mean * 1.10:
                print(f"  ⚠  TTFT climbing ({first_half_mean:.1f}ms → {last_half_mean:.1f}ms). "
                      f"Possible thermal throttle — using passes 2–3 for average.")
                warm = results[1:3]
            else:
                warm = results[1:]  # normal path: discard only warmup pass
        else:
            warm = results[1:] if len(results) > 1 else results

        return {
            "ttft_ms":          np.mean([r["ttft_ms"]          for r in warm]),
            "tps":              np.mean([r["tps"]               for r in warm]),
            "total_latency_ms": np.mean([r["total_latency_ms"]  for r in warm]),
            "generated_tokens": np.mean([r["generated_tokens"]  for r in warm]),
        }


In [6]:

# ── Quality Evaluator ──────────────────────────────────────────────────────────

class QualityEvaluator:
    """
    Scores INT8/INT4 responses relative to FP16 baseline using:
    - Cosine similarity of sentence-transformer embeddings (primary)
    - Jaccard word overlap (fallback if sentence-transformers missing)
    - Structural signals: JSON/table/bullet detection, completeness

    Known limitation: cosine similarity saturates quickly and does not
    catch factual degradation or hallucinations — only semantic drift
    from the FP16 reference output.
    """

    def __init__(self, use_sentence_transformers: bool = True):
        self.encoder = None
        if use_sentence_transformers:
            try:
                from sentence_transformers import SentenceTransformer
                self.encoder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu" )
                print("  ✓ Sentence-transformers loaded for semantic scoring")
            except ImportError:
                print("  ⚠ sentence-transformers not installed — falling back to Jaccard similarity")

    def score_response(self, response: str, prompt: str, baseline: str = None) -> dict:
        import re
        has_json    = bool(re.search(r'\{.*?\}', response, re.DOTALL))
        has_table   = "|" in response and "---" in response
        has_bullets = bool(re.search(r'^\s*[-*•]', response, re.MULTILINE))
        has_code    = "```" in response or "    " in response[:50]
        is_complete = (
            response.strip().endswith((".", "!", "?", "```", "}"))
            or len(response.split()) > 30
        )
        formatting_ok = True
        if "json"   in prompt.lower() and not has_json:    formatting_ok = False
        if "table"  in prompt.lower() and not has_table:   formatting_ok = False
        if "bullet" in prompt.lower() and not has_bullets: formatting_ok = False

        semantic_score = 1.0
        if baseline:
            if self.encoder:
                eb  = self.encoder.encode([baseline])
                ec  = self.encoder.encode([response])
                cos = float(np.dot(eb, ec.T) / (np.linalg.norm(eb) * np.linalg.norm(ec)))
                semantic_score = max(0.0, min(1.0, cos))
            else:
                a = set(baseline.lower().split())
                b = set(response.lower().split())
                semantic_score = len(a & b) / max(len(a | b), 1)

        return {
            "response_length": len(response.split()),
            "has_structure":   has_json or has_table or has_bullets or has_code,
            "is_complete":     is_complete,
            "semantic_score":  round(semantic_score, 4),
            "formatting_ok":   formatting_ok,
        }

    def run_suite(self, runner, precision, baseline_responses=None, max_per_category=3):
        """
        Run quality eval across all prompt categories.

        For FP16: caches responses as baseline while scoring (single inference pass).
        For INT8/INT4: scores against the cached FP16 baselines.
        Returns (results_list, updated_baseline_responses).
        """
        results             = []
        collected_baselines = baseline_responses or {}

        with torch.no_grad():
            for category, prompts in PROMPT_SUITE.items():
                print(f"  [{precision}] Category: {category}")

                if precision == "FP16":
                    collected_baselines[category] = {}

                for idx, prompt in enumerate(prompts[:max_per_category]):
                    r = runner.run(prompt)

                    # cache FP16 response as baseline during the same pass —
                    # avoids running FP16 prompts twice
                    if precision == "FP16":
                        collected_baselines[category][idx] = r["response"]

                    baseline = collected_baselines.get(category, {}).get(idx)
                    scores   = self.score_response(r["response"], prompt, baseline)

                    results.append({
                        "precision":  precision,
                        "category":   category,
                        "prompt_idx": idx,
                        "prompt":     prompt[:80] + "...",
                        "response":   r["response"],
                        **scores,
                    })

        return results, collected_baselines




In [7]:
# ── Benchmark Orchestrator ─────────────────────────────────────────────────────

class QuantizationBenchmark:
    def __init__(
        self,
        model_key:                str  = "TinyLlama-1.1B",
        precisions:               list = None,
        n_perf_runs:              int  = 5,
        max_quality_per_category: int  = 3,
        output_dir:               str  = "./benchmark_results",
    ):
        self.model_key   = model_key
        self.precisions  = precisions or ["FP16", "INT8", "INT4"]
        self.n_perf_runs = n_perf_runs
        self.max_quality = max_quality_per_category
        self.output_dir  = output_dir
        self.loader      = ModelLoader(model_key)
        self.evaluator   = QualityEvaluator()
        os.makedirs(output_dir, exist_ok=True)
        self.perf_results:       list = []
        self.quality_results:    list = []
        self.baseline_responses: dict = {}

    def run(self):
        for precision in self.precisions:
            print(f"\n{'#'*60}\n  PRECISION: {precision}\n{'#'*60}")

            self.loader.load(precision)

            # read clean model-only VRAM captured right after load in ModelLoader
            model_vram_mb = self.loader._model_vram_mb

            runner = InferenceRunner(self.loader.model, self.loader.tokenizer)
            perf   = MetricsCollector.benchmark_performance(runner, self.n_perf_runs)

            self.perf_results.append({
                "precision":        precision,
                "model":            self.model_key,
                "ttft_ms":          round(perf["ttft_ms"], 2),
                "tps":              round(perf["tps"], 2),
                "total_latency_ms": round(perf["total_latency_ms"], 2),
                "vram_mb":          round(model_vram_mb, 1),
                "generated_tokens": int(perf["generated_tokens"]),
            })

            # run_suite handles baseline caching internally for FP16 —
            # no separate _collect_baseline pass needed
            quality_rows, self.baseline_responses = self.evaluator.run_suite(
                runner, precision, self.baseline_responses, self.max_quality
            )
            self.quality_results.extend(quality_rows)

            self.loader.unload()

        self._save()
        self._summary()
        return self.perf_results, self.quality_results

    def _save(self):
        with open(os.path.join(self.output_dir, "perf_results.json"), "w") as f:
            json.dump(self.perf_results, f, indent=2)
        with open(os.path.join(self.output_dir, "quality_results.json"), "w") as f:
            json.dump(
                [{k: v for k, v in r.items() if k != "response"} for r in self.quality_results],
                f, indent=2,
            )
        print(f"\n✓ Results saved to {self.output_dir}/")

    def _summary(self):
        print("\n" + "="*60 + "\nBENCHMARK SUMMARY\n" + "="*60)
        df = pd.DataFrame(self.perf_results)
        print(df[["precision", "vram_mb", "ttft_ms", "tps", "total_latency_ms"]].to_string(index=False))
        qdf = pd.DataFrame([{k: v for k, v in r.items() if k != "response"} for r in self.quality_results])
        if not qdf.empty:
            print("\nQuality Summary:")
            print(
                qdf.groupby("precision")
                   .agg(semantic_score=("semantic_score", "mean"),
                        formatting_ok=("formatting_ok", "mean"),
                        is_complete=("is_complete", "mean"))
                   .round(3).to_string()
            )



In [8]:
# ── Entry Point ────────────────────────────────────────────────────────────────

def run_full_benchmark(
    model_key:   str  = "TinyLlama-1.1B",
    precisions:  list = None,
    n_perf_runs: int  = 5,
    output_dir:  str  = "./benchmark_results",
):
    if not torch.cuda.is_available():
        print("⚠  No CUDA detected — VRAM stats will be 0. Run on a GPU.")
    return QuantizationBenchmark(
        model_key=model_key,
        precisions=precisions or ["FP16", "INT8", "INT4"],
        n_perf_runs=n_perf_runs,
        output_dir=output_dir,
    ).run()

In [9]:
perf_results = run_full_benchmark()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ Sentence-transformers loaded for semantic scoring

############################################################
  PRECISION: FP16
############################################################

Loading TinyLlama-1.1B @ FP16


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model VRAM: 965.1 MB
  Running 5 inference passes...
    Pass 1: TTFT=678.5ms  TPS=24.4  Tokens=200
    Pass 2: TTFT=44.0ms  TPS=26.9  Tokens=200
    Pass 3: TTFT=42.1ms  TPS=26.9  Tokens=200
    Pass 4: TTFT=42.0ms  TPS=27.2  Tokens=200
    Pass 5: TTFT=43.8ms  TPS=26.9  Tokens=200
  [FP16] Category: reasoning
  [FP16] Category: summarization
  [FP16] Category: structured_output
  [FP16] Category: sql
  [FP16] Category: long_context
  [FP16] Category: hallucination_sensitive

############################################################
  PRECISION: INT8
############################################################

Loading TinyLlama-1.1B @ INT8


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model VRAM: 468.6 MB
  Running 5 inference passes...
    Pass 1: TTFT=174.7ms  TPS=8.0  Tokens=200
    Pass 2: TTFT=160.7ms  TPS=8.3  Tokens=200
    Pass 3: TTFT=150.6ms  TPS=8.3  Tokens=200
    Pass 4: TTFT=157.2ms  TPS=8.3  Tokens=200
    Pass 5: TTFT=156.2ms  TPS=8.3  Tokens=200
  [INT8] Category: reasoning
  [INT8] Category: summarization
  [INT8] Category: structured_output
  [INT8] Category: sql
  [INT8] Category: long_context
  [INT8] Category: hallucination_sensitive

############################################################
  PRECISION: INT4
############################################################

Loading TinyLlama-1.1B @ INT4


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model VRAM: 744.0 MB
  Running 5 inference passes...
    Pass 1: TTFT=92.0ms  TPS=16.9  Tokens=200
    Pass 2: TTFT=85.5ms  TPS=16.8  Tokens=200
    Pass 3: TTFT=84.5ms  TPS=16.9  Tokens=200
    Pass 4: TTFT=88.8ms  TPS=16.9  Tokens=200
    Pass 5: TTFT=84.0ms  TPS=16.8  Tokens=200
  [INT4] Category: reasoning
  [INT4] Category: summarization
  [INT4] Category: structured_output
  [INT4] Category: sql
  [INT4] Category: long_context
  [INT4] Category: hallucination_sensitive

✓ Results saved to ./benchmark_results/

BENCHMARK SUMMARY
precision  vram_mb  ttft_ms   tps  total_latency_ms
     FP16    965.1    42.98 26.96           7417.34
     INT8    468.6   156.17  8.28          24161.22
     INT4    744.0    85.72 16.86          11862.14

Quality Summary:
           semantic_score  formatting_ok  is_complete
precision                                            
FP16                1.000          0.833          1.0
INT4                0.845          0.778          1.0
INT8              

In [10]:
import pandas as pd

# perf_results is a tuple: (list_of_perf_dicts, list_of_quality_dicts)
# Unpack clearly to avoid indexing confusion
perf_list, quality_list = perf_results

df_perf = pd.DataFrame(perf_list)
print("\nPerformance Results:")
print(df_perf[["precision", "vram_mb", "ttft_ms", "tps", "total_latency_ms"]].to_string(index=False))

# quality_list is already a flat list of per-prompt dicts — iterate it directly.
# The previous code did `quality_results[1]` which grabbed only the 2nd element
# (a single dict) instead of the whole list, causing an empty or 1-row DataFrame.
df_qual = pd.DataFrame([{k: v for k, v in r.items() if k != "response"} for r in quality_list])
print("\nQuality Summary by Precision:")
print(df_qual.groupby("precision")[["semantic_score", "formatting_ok", "is_complete"]].mean().round(3))



Performance Results:
precision  vram_mb  ttft_ms   tps  total_latency_ms
     FP16    965.1    42.98 26.96           7417.34
     INT8    468.6   156.17  8.28          24161.22
     INT4    744.0    85.72 16.86          11862.14

Quality Summary by Precision:
           semantic_score  formatting_ok  is_complete
precision                                            
FP16                1.000          0.833          1.0
INT4                0.845          0.778          1.0
INT8                0.878          0.722          1.0


In [11]:
"""
LLM Quantization Tradeoff Analyzer — Gradio UI
================================================
Run after quantization_benchmark.py has produced results.
Or run with demo mode (synthetic data) if no GPU results yet.

Launch:
    python gradio_app.py
    python gradio_app.py --demo        # uses synthetic data
    python gradio_app.py --results ./benchmark_results
"""

import os, json, argparse
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import gradio as gr

# ── Color palette ──────────────────────────────────────────────────────────────
COLORS = {
    "FP16": "#4C9BE8",   # cool blue
    "INT8": "#F5A623",   # amber
    "INT4": "#7ED321",   # green
    "bg":   "#0F1117",
    "fg":   "#E8EAF0",
    "grid": "#2A2D3A",
}

plt.rcParams.update({
    "figure.facecolor":  COLORS["bg"],
    "axes.facecolor":    "#1A1D2E",
    "axes.edgecolor":    COLORS["grid"],
    "axes.labelcolor":   COLORS["fg"],
    "xtick.color":       COLORS["fg"],
    "ytick.color":       COLORS["fg"],
    "text.color":        COLORS["fg"],
    "grid.color":        COLORS["grid"],
    "grid.alpha":        0.4,
    "font.family":       "monospace",
})

# ── Synthetic demo data ────────────────────────────────────────────────────────

DEMO_PERF = [
    {"precision":"FP16","model":"TinyLlama-1.1B","ttft_ms":312,"tps":48.2,"total_latency_ms":4500,"vram_mb":2240,"load_time_s":8.1,"generated_tokens":190},
    {"precision":"INT8","model":"TinyLlama-1.1B","ttft_ms":248,"tps":61.5,"total_latency_ms":3530,"vram_mb":1340,"load_time_s":9.4,"generated_tokens":192},
    {"precision":"INT4","model":"TinyLlama-1.1B","ttft_ms":221,"tps":74.8,"total_latency_ms":2960,"vram_mb":780, "load_time_s":10.2,"generated_tokens":188},
]

DEMO_QUALITY_SUMMARY = {
    "FP16": {"semantic_score":1.00, "formatting_ok":0.93, "is_complete":0.97},
    "INT8": {"semantic_score":0.94, "formatting_ok":0.90, "is_complete":0.95},
    "INT4": {"semantic_score":0.87, "formatting_ok":0.83, "is_complete":0.91},
}

DEMO_CATEGORY_QUALITY = {
    "reasoning":           {"FP16":0.96,"INT8":0.91,"INT4":0.82},
    "summarization":       {"FP16":0.98,"INT8":0.95,"INT4":0.91},
    "structured_output":   {"FP16":0.93,"INT8":0.89,"INT4":0.79},
    "sql":                 {"FP16":0.97,"INT8":0.93,"INT4":0.85},
    "long_context":        {"FP16":0.95,"INT8":0.92,"INT4":0.88},
    "hallucination_sensitive":{"FP16":0.91,"INT8":0.87,"INT4":0.74},
}

# ── Data loader ────────────────────────────────────────────────────────────────

def load_results(results_dir: str):
    perf_path = os.path.join(results_dir, "perf_results.json")
    qual_path = os.path.join(results_dir, "quality_results.json")

    if not os.path.exists(perf_path):
        return None, None

    with open(perf_path) as f:
        perf = json.load(f)

    quality_summary = {p: {"semantic_score":0,"formatting_ok":0,"is_complete":0}
                       for p in ["FP16","INT8","INT4"]}
    category_quality = {}

    if os.path.exists(qual_path):
        with open(qual_path) as f:
            qual_raw = json.load(f)
        df = pd.DataFrame(qual_raw)
        if not df.empty:
            for prec, grp in df.groupby("precision"):
                quality_summary[prec] = {
                    "semantic_score": grp["semantic_score"].mean(),
                    "formatting_ok":  grp["formatting_ok"].mean(),
                    "is_complete":    grp["is_complete"].mean(),
                }
            for cat, grp in df.groupby("category"):
                category_quality[cat] = {
                    prec: sub["semantic_score"].mean()
                    for prec, sub in grp.groupby("precision")
                }

    return perf, (quality_summary, category_quality)


# ── Chart generators ───────────────────────────────────────────────────────────

def fig_perf_bars(perf):
    precs  = [p["precision"] for p in perf]
    colors = [COLORS[p] for p in precs]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle("Performance Metrics by Precision", fontsize=13, fontweight="bold", y=1.02)

    metrics = [
        ("vram_mb",          "VRAM Usage (MB)",       True),
        ("ttft_ms",          "Time to First Token (ms)", True),
        ("tps",              "Tokens per Second",      False),
    ]

    for ax, (key, label, lower_better) in zip(axes, metrics):
        vals = [p[key] for p in perf]
        bars = ax.bar(precs, vals, color=colors, width=0.5, edgecolor="none", alpha=0.9)

        # Annotate bars
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.02,
                    f"{val:.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

        # Baseline relative annotation
        baseline = vals[0]
        for i, (bar, val) in enumerate(zip(bars, vals)):
            if i > 0:
                delta = ((val - baseline) / baseline) * 100
                sign  = "+" if delta > 0 else ""
                color = "#7ED321" if (delta < 0 and lower_better) or (delta > 0 and not lower_better) else "#E85454"
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
                        f"{sign}{delta:.0f}%", ha="center", va="center",
                        fontsize=8, color=color, fontweight="bold")

        ax.set_title(label, fontsize=10, pad=8)
        ax.set_ylim(0, max(vals) * 1.25)
        ax.grid(axis="y", alpha=0.3)
        ax.spines[["top","right","left"]].set_visible(False)

    plt.tight_layout()
    return fig


def fig_vram_breakdown(perf):
    fig, ax = plt.subplots(figsize=(8, 4))

    precs = [p["precision"] for p in perf]
    vrams = [p["vram_mb"] for p in perf]
    colors = [COLORS[p] for p in precs]

    bars = ax.barh(precs, vrams, color=colors, height=0.45, edgecolor="none")
    ax.set_xlabel("VRAM (MB)", fontsize=10)
    ax.set_title("VRAM Footprint — Memory Bottleneck Analysis", fontsize=11, fontweight="bold")

    base = vrams[0]
    for bar, val, prec in zip(bars, vrams, precs):
        reduction = (1 - val/base) * 100
        label = f"{val:.0f} MB" + (f"  ↓{reduction:.0f}% vs FP16" if val < base else "  (baseline)")
        ax.text(val + base*0.01, bar.get_y() + bar.get_height()/2,
                label, va="center", fontsize=9)

    # Annotate compute vs memory bottleneck region
    ax.axvline(x=vrams[0]*0.6, color="#E85454", linestyle="--", alpha=0.6, linewidth=1)
    ax.text(vrams[0]*0.61, 0.05, "memory-bound\n← threshold", color="#E85454",
            fontsize=7, transform=ax.get_xaxis_transform(), va="bottom")

    ax.set_xlim(0, base * 1.3)
    ax.grid(axis="x", alpha=0.3)
    ax.spines[["top","right","bottom"]].set_visible(False)
    plt.tight_layout()
    return fig


def fig_quality_radar(quality_summary):
    categories = ["semantic_score", "formatting_ok", "is_complete"]
    labels     = ["Semantic\nConsistency", "Formatting\nStability", "Response\nCompleteness"]
    N = len(categories)

    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.set_facecolor("#1A1D2E")
    fig.patch.set_facecolor(COLORS["bg"])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, size=9)
    ax.set_ylim(0.6, 1.0)
    ax.set_yticks([0.7, 0.8, 0.9, 1.0])
    ax.set_yticklabels(["0.7","0.8","0.9","1.0"], size=7)
    ax.grid(color=COLORS["grid"], alpha=0.5)

    for prec, scores in quality_summary.items():
        vals = [scores[c] for c in categories]
        vals += vals[:1]
        ax.plot(angles, vals, linewidth=2, color=COLORS[prec], label=prec)
        ax.fill(angles, vals, color=COLORS[prec], alpha=0.15)

    ax.set_title("Quality Tradeoff Radar", fontsize=11, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=9)
    return fig


def fig_category_heatmap(category_quality):
    cats  = list(category_quality.keys())
    precs = ["FP16", "INT8", "INT4"]
    data  = np.array([[category_quality[c].get(p, np.nan) for p in precs] for c in cats])

    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(data, cmap="RdYlGn", vmin=0.65, vmax=1.0, aspect="auto")

    ax.set_xticks(range(len(precs)))
    ax.set_xticklabels(precs, fontsize=10)
    ax.set_yticks(range(len(cats)))
    ax.set_yticklabels([c.replace("_", " ").title() for c in cats], fontsize=9)
    ax.set_title("Semantic Score by Category & Precision", fontsize=11, fontweight="bold")

    for i in range(len(cats)):
        for j in range(len(precs)):
            v = data[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        fontsize=9, fontweight="bold",
                        color="black" if v > 0.82 else "white")

    plt.colorbar(im, ax=ax, shrink=0.8, label="Semantic Score")
    plt.tight_layout()
    return fig


def fig_tradeoff_scatter(perf, quality_summary):
    fig, ax = plt.subplots(figsize=(8, 5))

    for p in perf:
        prec = p["precision"]
        x    = p["tps"]
        y    = quality_summary[prec]["semantic_score"]
        size = 1000 / (p["vram_mb"] / 500 + 0.5)   # bubble ~ inverse VRAM

        ax.scatter(x, y, s=size, color=COLORS[prec], alpha=0.85,
                   edgecolors="white", linewidths=1.5, zorder=3)
        ax.annotate(f"{prec}\n{p['vram_mb']:.0f}MB",
                    (x, y), textcoords="offset points", xytext=(10, 5),
                    fontsize=9, color=COLORS[prec], fontweight="bold")

    ax.set_xlabel("Throughput (Tokens / second)", fontsize=10)
    ax.set_ylabel("Semantic Quality Score", fontsize=10)
    ax.set_title("Speed vs Quality Tradeoff  (bubble size ∝ 1/VRAM)", fontsize=11, fontweight="bold")

    # Pareto annotation
    ax.text(0.98, 0.05, "↑ Better quality\n→ Higher throughput",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=8, color=COLORS["fg"], alpha=0.6)

    ax.grid(alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    return fig


def fig_latency_decomposition(perf):
    """TTFT vs generation latency decomposition."""
    fig, ax = plt.subplots(figsize=(9, 4))

    precs = [p["precision"] for p in perf]
    ttfts = [p["ttft_ms"] for p in perf]
    rest  = [p["total_latency_ms"] - p["ttft_ms"] for p in perf]

    x = np.arange(len(precs))
    w = 0.4

    b1 = ax.bar(x, ttfts, w, label="TTFT (prefill)", color=[COLORS[p] for p in precs], alpha=0.9)
    b2 = ax.bar(x, rest,  w, bottom=ttfts, label="Generation latency",
                color=[COLORS[p] for p in precs], alpha=0.45, hatch="///")

    ax.set_xticks(x)
    ax.set_xticklabels(precs)
    ax.set_ylabel("Latency (ms)", fontsize=10)
    ax.set_title("Latency Decomposition: Prefill vs Generation", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    return fig


# ── Key Findings Generator ─────────────────────────────────────────────────────

def generate_findings(perf, quality_summary):
    fp16 = next(p for p in perf if p["precision"] == "FP16")
    int4 = next((p for p in perf if p["precision"] == "INT4"), None)
    int8 = next((p for p in perf if p["precision"] == "INT8"), None)

    findings = []

    if int4:
        vram_red = (1 - int4["vram_mb"] / fp16["vram_mb"]) * 100
        tps_gain = (int4["tps"] / fp16["tps"] - 1) * 100
        qual_drop = (1 - quality_summary["INT4"]["semantic_score"]) * 100

        findings.append(
            f"**VRAM Reduction (INT4):** {vram_red:.0f}% reduction vs FP16 "
            f"({fp16['vram_mb']:.0f}MB → {int4['vram_mb']:.0f}MB). "
            f"This is the dominant win for memory-constrained deployments."
        )
        findings.append(
            f"**Throughput Gain (INT4):** +{tps_gain:.0f}% TPS vs FP16 "
            f"({fp16['tps']:.1f} → {int4['tps']:.1f} tok/s). "
            f"Gains are real but smaller than VRAM reduction suggests — "
            f"dequantization overhead partially offsets compute savings."
        )
        findings.append(
            f"**Quality Cost (INT4):** Semantic drift of ~{qual_drop:.1f}% on average. "
            f"Hallucination-sensitive tasks show the steepest degradation — "
            f"not recommended for fact-critical applications without validation."
        )

    if int8:
        vram_red_8 = (1 - int8["vram_mb"] / fp16["vram_mb"]) * 100
        qual_drop_8 = (1 - quality_summary["INT8"]["semantic_score"]) * 100
        findings.append(
            f"**INT8 Sweet Spot:** {vram_red_8:.0f}% VRAM reduction with only "
            f"~{qual_drop_8:.1f}% quality drop. Best precision/quality tradeoff "
            f"for production systems that can't afford quality regression."
        )

    findings.append(
        "**Compute vs Memory Bottleneck:** At INT4, inference shifts from "
        "memory-bandwidth-bound to partially compute-bound (dequantization adds ALU load). "
        "This is why latency gains plateau faster than VRAM gains."
    )
    findings.append(
        "**Recommendation:** Use INT8 for production LLM APIs. "
        "Use INT4 only for edge/mobile deployments or when VRAM is the hard constraint. "
        "FP16 remains the baseline for quality-critical or long-form generation tasks."
    )

    return "\n\n".join(f"{i+1}. {f}" for i, f in enumerate(findings))


def build_tradeoff_table(perf, quality_summary):
    rows = []
    fp16_perf = next(p for p in perf if p["precision"] == "FP16")

    for p in perf:
        prec = p["precision"]
        vram_delta = f"−{(1-p['vram_mb']/fp16_perf['vram_mb'])*100:.0f}%" if prec != "FP16" else "baseline"
        tps_delta  = f"+{(p['tps']/fp16_perf['tps']-1)*100:.0f}%"         if prec != "FP16" else "baseline"
        qual = quality_summary[prec]["semantic_score"]

        rows.append({
            "Precision": prec,
            "VRAM (MB)": f"{p['vram_mb']:.0f}",
            "VRAM Δ":    vram_delta,
            "TPS":       f"{p['tps']:.1f}",
            "TPS Δ":     tps_delta,
            "TTFT (ms)": f"{p['ttft_ms']:.0f}",
            "Semantic Score": f"{qual:.3f}",
            "Rec. Use Case": {
                "FP16": "Quality-critical, research",
                "INT8": "Production APIs",
                "INT4": "Edge / VRAM-constrained",
            }[prec],
        })

    return pd.DataFrame(rows)


# ── Gradio App ─────────────────────────────────────────────────────────────────

def build_app(results_dir: str = None, demo_mode: bool = False):
    # Load or use demo data
    if demo_mode or results_dir is None:
        perf = DEMO_PERF
        quality_summary, category_quality = DEMO_QUALITY_SUMMARY, DEMO_CATEGORY_QUALITY
        data_source = "Demo data (synthetic). Run `quantization_benchmark.py` to get real GPU results."
    else:
        loaded_perf, loaded_qual = load_results(results_dir)
        if loaded_perf is None:
            perf = DEMO_PERF
            quality_summary, category_quality = DEMO_QUALITY_SUMMARY, DEMO_CATEGORY_QUALITY
            data_source = "No results found in directory. Showing demo data."
        else:
            perf = loaded_perf
            quality_summary, category_quality = loaded_qual
            data_source = f"Real GPU results loaded from `{results_dir}`"

    findings    = generate_findings(perf, quality_summary)
    tradeoff_df = build_tradeoff_table(perf, quality_summary)

    css = """
    .gradio-container { background: #0F1117; }
    .tab-nav { background: #1A1D2E !important; }
    h1, h2, h3 { color: #E8EAF0 !important; }
    .markdown-body { color: #C8CAD4 !important; }
    """

    with gr.Blocks(title="LLM Quantization Tradeoff Analyzer", css=css, theme=gr.themes.Default()) as app:

        gr.Markdown(f"""
# ⚡ LLM Quantization Tradeoff Analyzer
**Measuring FP16 / INT8 / INT4 across performance, quality, and systems efficiency**

_{data_source}_
""")

        with gr.Tabs():

            # ── Tab 1: Performance ──
            with gr.Tab("Performance"):
                gr.Markdown("### Inference Performance Across Precisions")
                gr.Plot(value=fig_perf_bars(perf))
                gr.Plot(value=fig_latency_decomposition(perf))
                gr.Plot(value=fig_vram_breakdown(perf))

            # ── Tab 2: Quality ──
            with gr.Tab("Quality"):
                gr.Markdown("### Response Quality vs Baseline (FP16)")
                with gr.Row():
                    gr.Plot(value=fig_quality_radar(quality_summary))
                    gr.Plot(value=fig_category_heatmap(category_quality))

            # ── Tab 3: Tradeoff ──
            with gr.Tab("Tradeoff Analysis"):
                gr.Markdown("### Speed vs Quality vs Memory — The Core Tradeoff")
                gr.Plot(value=fig_tradeoff_scatter(perf, quality_summary))

                gr.Markdown("### Summary Matrix")
                gr.Dataframe(
                    value=tradeoff_df,
                    interactive=False,
                    wrap=True,
                )

            # ── Tab 4: Findings ──
            with gr.Tab("Key Findings"):
                gr.Markdown(f"""
### Engineering Findings

{findings}

---
### Systems Insight: Compute vs Memory Bottleneck

| Precision | Primary Bottleneck | Implication |
|---|---|---|
| FP16 | Memory bandwidth | Throughput scales with HBM bandwidth |
| INT8 | Memory bandwidth (lighter) | ~2× effective bandwidth, minimal overhead |
| INT4 | Mixed (compute + memory) | Dequantization adds ALU cost; gains plateau |

> *"INT4 provided the largest VRAM reduction, but latency gains were smaller than expected
> because inference shifted to partially compute-bound — dequantization overhead
> on the GPU's ALU units partially offsets memory bandwidth savings."*
""")

            # ── Tab 5: Live Inference ──
            with gr.Tab("Live Inference"):
                gr.Markdown("""
### Run Live Inference
*This requires a GPU session with models loaded. Best used on Kaggle/Colab.*
""")
                gr.Markdown("""
**To enable live inference:**
1. Run `quantization_benchmark.py` on Kaggle/Colab (GPU)
2. Modify this tab to call your loaded model directly
3. Or use the Kaggle notebook version which has live inference built in
""")

    return app

# if __name__ == "__main__":
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--demo",    action="store_true",  help="Use synthetic demo data")
#     parser.add_argument("--results", type=str, default=None, help="Path to benchmark_results dir")
#     parser.add_argument("--port",    type=int, default=7860)
#     parser.add_argument("--share",   action="store_true",  help="Create public Gradio link")
#     args = parser.parse_args()

#     app = build_app(results_dir=args.results, demo_mode=args.demo)
#     app.launch(server_port=args.port, share=args.share, show_error=True)

In [13]:
# from app import build_app
app = build_app(results_dir="./benchmark_results", demo_mode=False)
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://031993510c47a0cad6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
import shutil
shutil.make_archive("quantization_results", "zip", "./benchmark_results")
print("Download: Files panel → quantization_results.zip")

Download: Files panel → quantization_results.zip


### Engineering Findings
- **VRAM Reduction (INT8):** 51% reduction vs FP16 (965MB → 469MB). Dominant win for memory-constrained deployments — clean 1 byte/weight storage as expected.
- **VRAM Anomaly (INT4):** Only 23% reduction vs FP16 (965MB → 744MB), worse than INT8. Caused by bitsandbytes double-quantization metadata overhead inflating measured footprint. True weight-only footprint should be ~241MB (~75% reduction).
- **Throughput Regression (INT4/INT8):** FP16 at 27 TPS outperforms both INT4 (17 TPS, −37%) and INT8 (8 TPS, −69%). On T4 (memory-bandwidth-bound, no INT4 tensor cores), bitsandbytes dequantization overhead fully negates compute savings. Throughput gains require hardware with native INT8/INT4 ALUs (A100, H100, MI300X).
- **Quality Cost (INT4):** Semantic drift of ~15.5% vs FP16 baseline. INT8 shows less drift (~12.2%) while also using less VRAM — making INT4 the dominated choice on this hardware.
- **INT8 Sweet Spot:** 51% VRAM reduction with only 12.2% semantic quality drop and lowest absolute memory footprint. Clear winner on T4-class hardware without dedicated low-precision tensor cores.
- **Recommendation:** Use INT8 for production deployments on bandwidth-bound GPUs. INT4 gains only materialise on hardware with native quantized compute (use AutoAWQ/GPTQ on ROCm or TensorRT-LLM on Hopper+). FP16 remains baseline for quality-critical tasks.

### Systems Insight: Compute vs Memory Bottleneck

| Precision | Primary Bottleneck | Implication |
| :--- | :--- | :--- |
| FP16 | Memory bandwidth | Native CUDA kernels, no overhead — fastest on T4 |
| INT8 | Memory bandwidth (lighter) | ~2× effective bandwidth, but bitsandbytes dequant adds latency |
| INT4 | Compute-bound (dequant) | ALU overhead dominates on non-tensor-core hardware; gains plateau |